In [ ]:
import numpy as np
from rdkit import Chem
#audo-grading program for furanose (five-membered-ring)
n_ring=5 #five-membered ring
#Step 0: Compare non-isomeric canonical SMILES
def mol_to_smiles(mol_file_path):
    try:
        # read input Molfile by RDKit
        mol = Chem.MolFromMolFile(mol_file_path)
        if mol is None:
            raise ValueError("Error:Check file path and format.")
        smiles = Chem.MolToSmiles(mol,isomericSmiles=False, canonical=True) 
        return smiles 
    except Exception as e:
        print(f"Error when processing Molfile: {e}")
        return None
# 
#read answer key
if __name__ == "__main__":
    #V2000 Molfile path here
    key_mol_file = "5C_beta-D-fructose_key.mol"  #use β-D-fructose as an example
    key_canonical_smiles = mol_to_smiles(key_mol_file)
#
#get student's answer as input 
if __name__ == "__main__":
    input_mol_file = "5C_beta-D-fructose_input1.mol"  #input1-correct
    #input_mol_file = "5C_beta-D-fructose_input2.mol"  #input2-wrong
    input_smiles = mol_to_smiles(input_mol_file)
#
if input_smiles == key_canonical_smiles:
    mark = 1
else:
    mark = 0
print("non-isomeric SMILES match?",mark)
#Step1: parse Molfiles
def read_mol_file(file_path):
    atoms = []
    with open(file_path, 'r') as f:
        lines = f.readlines()
        try:
            num_atoms = int(lines[3][:3].strip())
        except (IndexError, ValueError):
            raise ValueError("invalid Molfile format")
               
        for i in range(num_atoms):
            line = lines[4 + i]
            try:
                x = float(line[:10].strip())
                y = float(line[10:20].strip())
                element = line[31:34].strip().capitalize()  
                atoms.append([x, y, element])
            except (IndexError, ValueError):
                raise ValueError(f"error in atom lines: {line.strip()}")                
    return atoms
#read Molfile of the key
if __name__ == "__main__":
    try:
        atom_data = read_mol_file(key_mol_file)
        molecule = np.array(atom_data, dtype=object)
        num_atom = molecule.shape[0]      
    except Exception as e:
        print(f"error when processing Molfile: {e}")
#read Molfile of the input
if __name__ == "__main__":
    try:
        atom_data = read_mol_file(input_mol_file)
        input = np.array(atom_data, dtype=object)        
    except Exception as e:
        print(f"error when processing Molfile: {e}")
#Step2: analyze and extract features of the key's Molfiles
#extract the feaure of the key
f_c1o=0 #initialization
f_c2o=0
f_c3o=0
akey_feature = [f_c1o,f_c2o,f_c3o]
#
ring = molecule[:n_ring]
x_values = [atom[0] for atom in ring]
sorted_x = sorted(x_values) 
y_values = [atom[1] for atom in ring]
sorted_y = sorted(y_values)
#
#C1
#
greatest_x = sorted_x[n_ring-1]
for i_c1, atom in enumerate(ring):
    if atom[0] == greatest_x:
        break
#print("molecule[i_c1][0] =", molecule[i_c1][0])
#print(i_c1)
#
x_c1 = molecule[i_c1][0]
#
# Find i_c1o in indices 6 to 11 satisfying both conditions
for i_c1o in range(n_ring, num_atom):
    x_c1o = molecule[i_c1o][0]
    if (x_c1 - 0.1 <= x_c1o <= x_c1 + 0.1) and molecule[i_c1o][2] == 'O':
        break
#print("Matching indices (i_c1o):", i_c1o)
#
if molecule[i_c1o][1] > molecule[i_c1][1]:
    f_c1o = 1
else:
    f_c1o = -1
#feature c1o: if the O on C1 is up, c1o=1, otherwise, c10=-1
#
#C2
#
forth_x = sorted_x[n_ring-2]
min_y = min(y_values)
sec_y = sorted_y[1]
#search for the index of C atom (C with x value ranked forth or higher)
i_c2 = None
for i, atom in enumerate(ring):
    x, y = atom[0], atom[1]
    if x == forth_x and (y == min_y or y==sec_y) and molecule[i][2] == 'C':
        i_c2 = i
        break
#if no satisfied atoms found, report error
if i_c2 is None:
    raise ValueError("No satisfied C atom found!")

x_c2 = molecule[i_c2][0]
# Find i_c1o in indices 6 to 11 satisfying both conditions
for i_c2o in range(n_ring, num_atom):
    x_c2o = molecule[i_c2o][0]
    if (x_c2 - 0.1 <= x_c2o <= x_c2 + 0.1) and molecule[i_c2o][2] == 'O':
        break
#print("Matching indices (i_c2o):", i_c2o)
#
if molecule[i_c2o][1] > molecule[i_c2][1]:
    f_c2o = 1
elif molecule[i_c2o][1] < molecule[i_c2][1]:
    f_c2o = -1
#
#C3
#
sec_x = sorted_x[1]
min_y = min(y_values) 
sec_y = sorted_y[1] 
#
for i_c3, atom in enumerate(ring):
    x, y = atom[0], atom[1]
    if x == sec_x and (y == min_y or y == sec_y) and molecule[i][2] == 'C':
        break  
x_c3 = molecule[i_c3][0]
# Find i_c3o in indices 6 to 11 satisfying both conditions
for i_c3o in range(n_ring, num_atom):
    x_c3o = molecule[i_c3o][0]
    if (x_c3 - 0.1 <= x_c3o <= x_c3 + 0.1) and molecule[i_c3o][2] == 'O':
        break
#print("Matching indices (i_c3o):", i_c3o)
#
if molecule[i_c3o][1] > molecule[i_c3][1]:
    f_c3o = 1
elif molecule[i_c3o][1] < molecule[i_c3][1]:
    f_c3o = -1
#update the feature value of the key
key_feature = [f_c1o,f_c2o,f_c3o]
print("key feature:", key_feature)
#
#Step3: extract features of the input's Molfiles and compare it with the key
ring = input[:n_ring]
x_values = [atom[0] for atom in ring]
sorted_x = sorted(x_values)  
y_values = [atom[1] for atom in ring]   
sorted_y = sorted(y_values)
#
#Step 3A，Haworth convention check1: top atom on the ring is O
#
third_x = sorted_x[2]
max_y = max(y_values)

found = False
for i_o, atom in enumerate(ring):
    x, y = atom[0], atom[1]
    if  y == max_y:
        found = True
        break  # Found the index
#print("i_o: ", i_o)
#print(found)
#
if not found:
    i_o = -1
#print("i_o: ", i_o)

if input[i_o][2] == 'O':
    mark = 1
else:
    mark = 0
#Step 3B, check Haworth convention2: the C on the upper right C in the ring points up
sec_x = sorted_x[1]
third_x = sorted_x[2]
third_y = sorted_y[2]
forth_y = sorted_y[3]

for i_c5, atom in enumerate(ring):
    x, y = atom[0], atom[1]
    if (x == sec_x or x == third_x) and (y == third_y or y == forth_y):
        break
#print("input[i_c5][0] =", input[i_c5][0])
#print("i_c5: ",i_c5)
x_c5 = input[i_c5][0]

found = False
# Search for atoms that satisfy the conditions:
for i_c5c in range(n_ring, num_atom):
    x_c5c = input[i_c5c][0]
    if (x_c5 - 0.1 <= x_c5c <= x_c5 + 0.1) and input[i_c5c][2] == 'C':  
        found = True
        break
# If no match was found, set x_c5 to -1
if not found:
    i_c5c = -1

#print("input[i_c5c][2]: ", input[i_c5c][2])
#print("Matching indices (i_c5c):", i_c5c)

if i_c5c >= 0:
    if input[i_c5c][1] > input[i_c5][1]:
        mark = 1
    else:
        mark = 0
else:
    mark = 0
#
#Step 3C: Check stereochemistry in Haworth projection
#five-membered ring，compare the -OH directions on C1-C3
f_c1o=0
f_c2o=0
f_c3o=0
input_feature = [f_c1o,f_c2o,f_c3o]
#
#C1
#
greatest_x = sorted_x[n_ring-1]
for i_c1, atom in enumerate(ring):
    if atom[0] == greatest_x and input[i_c1][2] == 'C':
        break
#print("input[i_c1][0] =", input[i_c1][0])
#print(i_c1)
#
x_c1 = input[i_c1][0]
#print(x_c1)
#
# Find i_c1o in indices 6 to 11 satisfying both conditions
for i_c1o in range(n_ring, num_atom):
    x_c1o = input[i_c1o][0]
    if (x_c1 - 0.1 <= x_c1o <= x_c1 + 0.1) and input[i_c1o][2] == 'O':
        break

#print("Matching indices (i_c1o):", i_c1o)
#
if input[i_c1o][1] > input[i_c1][1]:
    f_c1o = 1
else:
    f_c1o = -1
#
#C2
#
forth_x = sorted_x[n_ring-2]
i_c2 = None
for i, atom in enumerate(ring):
    x, y = atom[0], atom[1]
    if x == forth_x:
        i_c2 = i
        break
if i_c2 is None:
    raise ValueError("No satisfied C atom found!")
x_c2 = input[i_c2][0]
# Find i_c1o in indices 5 to num_atom satisfying both conditions
for i_c2o in range(n_ring, num_atom):
    x_c2o = input[i_c2o][0]
    if (x_c2 - 0.1 <= x_c2o <= x_c2 + 0.1) and input[i_c2o][2] == 'O':
        break

#print("Matching indices (i_c2o):", i_c2o)
#
if input[i_c2o][1] > input[i_c2][1]:
    f_c2o = 1
elif input[i_c2o][1] < input[i_c2][1]:
    f_c2o = -1
#
#C3
#
sec_x = sorted_x[1]
min_y = min(y_values)  
sec_y = sorted_y[1]
#
for i_c3, atom in enumerate(ring):
    x, y = atom[0], atom[1]
    if x == sec_x:
        break    
x_c3 = input[i_c3][0]
# Find i_c3o in indices 5 to num_atom satisfying both conditions
for i_c3o in range(n_ring, num_atom):
    x_c3o = input[i_c3o][0]
    if (x_c3 - 0.1 <= x_c3o <= x_c3 + 0.1) and input[i_c3o][2] == 'O':
        break

#print("Matching indices (i_c3o):", i_c3o)
#
if input[i_c3o][1] > input[i_c3][1]:
    f_c3o = 1
elif input[i_c3o][1] < input[i_c3][1]:
    f_c3o = -1
#
#feature value of the input
input_feature = [f_c1o,f_c2o,f_c3o]
#print("input feature:", input_feature)
#
if input_feature == key_feature:
    mark = 1
else:
    mark = 0
print("final mark is (correct=1, wrong=0):",mark)

**Cite this work**

> Li, X.; Hu, C.; Su, Y.; Lu, H. "StereoLearn: An Online Platform for Learning and Automatic Grading of Organic Structure Representations." *Journal of Chemical Education*, 2026.

If you use StereoLearn and associate content in your teaching or research, please cite the above publication.

📧 Contact: xiaoli@pku.edu.cn

Thank you!